<a href="https://colab.research.google.com/github/chewyiyu0225/Moonlight/blob/main/moonlight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Import Libraries

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder,StandardScaler, MultiLabelBinarizer
from google.colab import data_table
from sklearn.preprocessing import MultiLabelBinarizer

# Enable interactive tables in Google Colab
data_table.enable_dataframe_formatter()


2. Load Dataset

In [ ]:
df = pd.read_csv('/dating_data.csv')
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

# check columns names
# print(df.columns)

3. Feature Selection

In [ ]:
# Define the list of features required
required_features = [
    'gender', 'sexual_orientation', 'location_type', 'income_bracket',
    'education_level', 'interest_tags', 'app_usage_time_label',
    'swipe_right_label','likes_received', 'mutual_matches',
    'profile_pics_count', 'bio_length','message_sent_count',
    'emoji_usage_rate', 'last_active_hour', 'swipe_time_of_day',
    'match_outcome', 'age', 'height_cm', 'weight_kg',
    'zodiac_sign', 'body_type', 'relationship_intent']

# Create a copy of the dataframe with only the required columns
df_final = df[required_features].copy()

In [ ]:
# Feature Sets for project "Moonlight"
features_archetypes = ['app_usage_time_label', 'swipe_right_label', 'likes_received', 'message_sent_count', 'emoji_usage_rate', 'bio_length', 'profile_pics_count', 'last_active_hour']

features_maximizer = ['gender', 'age', 'height_cm', 'weight_kg', 'body_type', 'income_bracket', 'education_level', 'interest_tags', 'relationship_intent', 'zodiac_sign']

features_love_clock = ['match_outcome', 'location_type', 'message_sent_count', 'relationship_intent', 'interest_tags', 'age']

Define Categotization Logic

In [ ]:
# Reset 'interest_tags' from the original data to get the clean text
df_final['interest_tags'] = df['interest_tags'].copy()

# Define the mapping function to return a LIST of all matching categories
def get_category_list(tags):
    if pd.isna(tags): return ['General']
    tags = str(tags).lower()
    categories = []

    if any(word in tags for word in ['fitness', 'yoga', 'hiking', 'running', 'skating', 'mma', 'motorcycling']):
        categories.append('Active_Sports')
    if any(word in tags for word in ['fashion', 'movies', 'diy', 'painting', 'poetry', 'art', 'dancing', 'writing', 'crafting', 'photography', 'tattoos', 'makeup', 'anime', 'k-pop', 'binge-watching']):
        categories.append('Creative_Arts')
    if any(word in tags for word in ['coding', 'tech', 'startups', 'investing']):
        categories.append('Tech_Career')
    if any(word in tags for word in ['astrology', 'spirituality', 'meditation']):
        categories.append('Spiritual_Beliefs')
    if any(word in tags for word in ['politics', 'history', 'social activism', 'languages', 'podcasts']):
        categories.append('Intellectual_Society')
    if any(word in tags for word in ['traveling', 'parenting', 'cars', 'clubbing', 'sneaker culture', 'board games', 'memes', 'stand-up comedy', 'foodie', 'pets', 'gardening', 'reading', 'cooking']):
        categories.append('Social_Lifestyle')

    return categories if categories else ['General']

# Apply the function to get lists of categories
df_final['interest_list'] = df_final['interest_tags'].apply(get_category_list)

# Use MultiLabelBinarizer to create the binary matrix (0s and 1s)
mlb = MultiLabelBinarizer()
tag_matrix = mlb.fit_transform(df_final['interest_list'])

# Create a new DataFrame for these binary columns
df_tags_binary = pd.DataFrame(tag_matrix, columns=mlb.classes_)

# CMerge new columns and drop the old text column
df_final = pd.concat([df_final.reset_index(drop=True), df_tags_binary], axis=1)
df_final = df_final.drop(columns=['interest_tags', 'interest_list'])

4. Handling Missing Value

In [ ]:
# Identify numeric columns for median imputation
numeric_cols = ['age', 'height_cm', 'weight_kg']
# Fill missing numeric values with the median
df_final[numeric_cols] = df_final[numeric_cols].fillna(df_final[numeric_cols].median())

# Identify categorical columns for mode imputation
categorical_cols = [col for col in df_final.columns if col not in numeric_cols]
# Fill missing categorical values with the most frequent value (mode)
for col in categorical_cols:
    df_final[col] = df_final[col].fillna(df_final[col].mode()[0])

5. Categorical Encoding (Text to Numbers)

In [ ]:
# Initialize the LabelEncoder
encoder = LabelEncoder()

# Convert all text-based columns into numerical labels
for col in categorical_cols:
    df_final[col] = encoder.fit_transform(df_final[col].astype(str))

6. Feature Scaling

In [ ]:
# Initialize StandardScaler to normalize numeric data
scaler = StandardScaler()

# Apply scaling ONLY to numeric columns to ensure they have the same scale
df_final[numeric_cols] = scaler.fit_transform(df_final[numeric_cols])

7. Final Output

In [ ]:
# 4. Display
df_final

,gender,sexual_orientation,location_type,income_bracket,education_level,app_usage_time_label,swipe_right_label,likes_received,mutual_matches,profile_pics_count,...,weight_kg,zodiac_sign,body_type,relationship_intent,Active_Sports,Creative_Arts,Intellectual_Society,Social_Lifestyle,Spiritual_Beliefs,Tech_Career
0,4,3,5,0,1,5,2,83,16,4,...,-1.845150,10,2,2,1,0,1,1,0,0
1,2,1,4,4,6,2,2,10,28,3,...,-0.140532,5,4,3,0,1,1,1,0,0
2,3,5,4,1,5,5,2,192,20,2,...,1.452789,8,2,5,0,1,0,1,0,0
3,1,3,0,6,8,2,0,54,27,5,...,-1.335522,10,5,1,0,0,1,0,0,1
4,2,1,5,3,1,3,0,195,3,1,...,-1.880297,6,5,0,0,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,5,3,0,5,8,0,2,200,29,5,...,1.792541,7,4,0,0,1,0,0,0,0
49996,0,4,3,1,2,2,2,83,1,1,...,0.445248,5,0,2,1,0,0,1,0,0
49997,2,1,1,0,3,3,2,28,29,1,...,-0.175678,7,0,5,1,1,0,0,0,0
49998,5,6,5,1,4,3,0,56,11,3,...,0.673702,6,0,2,1,0,0,0,1,0


8. csv file after remove useless columns
df_final.to_csv('cleaned_data.csv', index=False)